In [19]:
import pandas as pd
import ast

from collections import Counter, defaultdict
import math
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [3]:
train_df = pd.read_csv("train_df.csv")
test_df = pd.read_csv("test_df.csv")

In [4]:
train_df

,tokens,category_10
0,"['united', 'states', 'high', 'school', 'gradua...",POLITICS
1,"['a', 'guide', 'to', 'unscrambling', 'trumps',...",POLITICS
2,"['watching', 'conan', 'sell', 'cheese', 'on', ...",OTHER
3,"['muggle', 'dads', 'homemade', 'diagon', 'alle...",OTHER
4,"['pandemic', 'products', 'that', 'definitely',...",STYLE & BEAUTY
...,...,...
167616,"['friendships', 'matter']",OTHER
167617,"['las', 'vegas', 'launches', 'gay', 'travel', ...",QUEER VOICES
167618,"['culture', 'contrasts', 'norways', 'support',...",OTHER
167619,"['daily', 'meditation', 'i', 'hear', 'america'...",OTHER


In [5]:
train_df["tokens"] = train_df["tokens"].apply(ast.literal_eval)
test_df["tokens"] = test_df["tokens"].apply(ast.literal_eval)

In [6]:
train_df

,tokens,category_10
0,"[united, states, high, school, graduation, rat...",POLITICS
1,"[a, guide, to, unscrambling, trumps, bonkers, ...",POLITICS
2,"[watching, conan, sell, cheese, on, a, mexican...",OTHER
3,"[muggle, dads, homemade, diagon, alley, will, ...",OTHER
4,"[pandemic, products, that, definitely, belong,...",STYLE & BEAUTY
...,...,...
167616,"[friendships, matter]",OTHER
167617,"[las, vegas, launches, gay, travel, campaign, ...",QUEER VOICES
167618,"[culture, contrasts, norways, support, of, wor...",OTHER
167619,"[daily, meditation, i, hear, america, singing]",OTHER


In [7]:
t = train_df.iloc[0]["tokens"]
print(t)
print(type(t))
print(type(t[0]))


['united', 'states', 'high', 'school', 'graduation', 'rate', 'reaches', 'a', 'record', 'high']
<class 'list'>
<class 'str'>


In [12]:
unigram_models = {}

categories = train_df["category_10"].unique()

for category in categories:
    cat_tokens = train_df[train_df["category_10"] == category]["tokens"]
    print(cat_tokens,'\n\n')
    all_tokens = [token for tokens in cat_tokens for token in tokens]
    print(all_tokens[:10],'\n\n')
    
    word_counts = Counter(all_tokens)
    total_tokens = sum(word_counts.values())
    vocab = set(word_counts.keys())
    
    unigram_models[category] = {
        "word_counts": word_counts,
        "total_tokens": total_tokens,
        "vocab": vocab,
        "vocab_size": len(vocab)
    }
    print(f"unigram_models{category} ,{unigram_models[category]}")


0         [united, states, high, school, graduation, rat...
1         [a, guide, to, unscrambling, trumps, bonkers, ...
12        [gop, lawmakers, to, trump, we, can, still, wo...
14        [elizabeth, warren, americans, should, keep, d...
15        [vulnerable, gop, senator, now, says, hes, wil...
                                ...                        
167601    [former, intel, officials, defend, john, brenn...
167606    [trump, scorns, biden, for, the, way, he, wear...
167610    [one, quote, from, abraham, lincoln, might, ju...
167614    [kentucky, neighbor, expected, to, plead, guil...
167620                          [escaping, from, the, dark]
Name: tokens, Length: 28481, dtype: object 


['united', 'states', 'high', 'school', 'graduation', 'rate', 'reaches', 'a', 'record', 'high'] 


unigram_modelsPOLITICS ,{'word_counts': Counter({'to': 8464, 'the': 7738, 'trump': 5918, 'of': 4419, 'in': 4177, 'for': 3936, 'a': 3837, '<': 3401, 'NUM': 3401, '>': 3401, 'on': 3284, 'is': 2583, 

In [13]:
unigram_models["POLITICS"]["word_counts"].most_common(10)

[('to', 8464),
 ('the', 7738),
 ('trump', 5918),
 ('of', 4419),
 ('in', 4177),
 ('for', 3936),
 ('a', 3837),
 ('<', 3401),
 ('NUM', 3401),
 ('>', 3401)]

In [22]:
unigram_models

{'POLITICS': {'word_counts': Counter({'to': 8464,
           'the': 7738,
           'trump': 5918,
           'of': 4419,
           'in': 4177,
           'for': 3936,
           'a': 3837,
           '<': 3401,
           'NUM': 3401,
           '>': 3401,
           'on': 3284,
           'is': 2583,
           'donald': 2502,
           'and': 2371,
           'trumps': 1872,
           'with': 1651,
           'gop': 1449,
           'clinton': 1239,
           'about': 1223,
           'says': 1182,
           'new': 1092,
           'his': 1091,
           'obama': 1061,
           'are': 1056,
           'house': 1053,
           'at': 1038,
           'from': 1036,
           'hillary': 1019,
           'as': 992,
           'not': 991,
           'us': 979,
           'over': 911,
           'be': 894,
           'after': 892,
           'it': 871,
           'what': 816,
           'will': 804,
           'that': 792,
           'white': 764,
           'how': 742,
        

In [14]:
def unigram_prob(word, model):
    return (model["word_counts"].get(word, 0) + 1) / (
        model["total_tokens"] + model["vocab_size"]
    )


In [15]:
def unigram_log_likelihood(tokens, model):
    log_prob = 0.0
    for word in tokens:
        prob = unigram_prob(word, model)
        log_prob += math.log(prob)
    return log_prob


In [16]:
def classify_unigram(tokens, unigram_models):
    scores = {}
    for category, model in unigram_models.items():
        scores[category] = unigram_log_likelihood(tokens, model)
    
    return max(scores, key=scores.get)


In [18]:
sample_tokens = test_df.iloc[0]["tokens"]
true_label = test_df.iloc[0]["category_10"]

pred_label = classify_unigram(sample_tokens, unigram_models)

print("Tokens:", sample_tokens)
print("True label:", true_label)
print("Predicted label:", pred_label)


Tokens: ['<', 'NUM', '>', 'body', 'language', 'blunders', 'that', 'make', 'you', 'look', 'bad']
True label: OTHER
Predicted label: WELLNESS


In [20]:
y_true = test_df["category_10"].tolist()
y_pred = []

for tokens in test_df["tokens"]:
    pred = classify_unigram(tokens, unigram_models)
    y_pred.append(pred)

accuracy = accuracy_score(y_true, y_pred)
print(f"Unigram LM Accuracy: {accuracy:.4f}")


Unigram LM Accuracy: 0.4799


In [21]:
cm = confusion_matrix(y_true, y_pred, labels=categories)
cm_df = pd.DataFrame(cm, index=categories, columns=categories)
print(cm_df)
report = classification_report(y_true, y_pred, labels=categories)
print(report)


                POLITICS  OTHER  STYLE & BEAUTY  WELLNESS  TRAVEL  \
POLITICS            5081    222              41       114      93   
OTHER               1595   3472             962      2412    1237   
STYLE & BEAUTY         8     17            1548        82      44   
WELLNESS              28     90              38      2463     138   
TRAVEL                12     26              34        36    1584   
ENTERTAINMENT        111     74             210        62      87   
SPORTS                 9     12              11        17      16   
FOOD & DRINK           1     14               8        35      50   
QUEER VOICES          34     22              11        43      22   
BUSINESS              67     24               8        90      49   

                ENTERTAINMENT  SPORTS  FOOD & DRINK  QUEER VOICES  BUSINESS  
POLITICS                  109     263            90           419       689  
OTHER                    1227    1466          1633          2519      2507  
STYLE 